# SeBot-MLS on Colab
Same Python project/model as the Windows baseline. Run preprocessing once,
persist its cache, then train. Do not present synthetic smoke scores as MGTAB results.
See docs/SPECIFICATION.md for unresolved paper choices. Kaggle is the recommended
optional cloud training environment; code is authored/tested locally in VS Code.
Select a GPU accelerator for training, and enable Internet for dependency setup.
The notebook uses one GPU, even if two T4s are allocated.


Upload the supplied `sebot-mls-baseline.zip` when prompted. Mount Drive to preserve datasets, cache and checkpoints across runtime resets.

In [ ]:
from google.colab import drive, files
drive.mount("/content/drive")
uploaded = files.upload()


In [ ]:
from pathlib import Path
import zipfile, os
archive = Path(next(name for name in uploaded if name.endswith('.zip')))
with zipfile.ZipFile(archive) as z:
    z.extractall('/content/sebot-source')
PROJECT = Path('/content/sebot-source/sebot-mls-baseline')
assert (PROJECT / 'pyproject.toml').exists()
PERSIST = Path('/content/drive/MyDrive/SNA-SeBot-MLS')
PERSIST.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT)


Install the verified package versions. The pinned runtime requires Python 3.10-3.12. If the hosted default changes, use a matching Python environment rather than silently changing the baseline dependencies. Installation may take several minutes; GPU quotas/session resources are managed by the provider.

In [ ]:
import sys, subprocess
assert (3, 10) <= sys.version_info[:2] <= (3, 12), 'Use Python 3.10-3.12'
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.5.1', '--index-url', 'https://download.pytorch.org/whl/cu121'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', '.'], check=True)


Restart the notebook kernel/runtime after installation if NumPy/PyTorch were already imported. Then rerun the path cell above, without repeating upload/extraction.

In [ ]:
import sys, subprocess, yaml
(PROJECT / 'tools').mkdir(exist_ok=True)
subprocess.run(['g++', '-O3', '-std=c++11', 'vendor/orca/orca.cpp', '-o', 'tools/orca'], check=True)
subprocess.run([sys.executable, '-m', 'sebot_mls.cli', 'doctor'], check=True)


In [ ]:
# Change RAW_DATA to a folder containing the five official MGTAB tensors,
# or leave it here and use the download cell below. Keep the same seed and splits.
RAW_DATA = PERSIST / 'data' / 'MGTAB'
CACHE = PERSIST / 'processed'
OUTPUT = PERSIST / 'outputs'
cfg = yaml.safe_load((PROJECT / 'configs/mgtab.yaml').read_text())
cfg.update(data_dir=str(RAW_DATA), cache_dir=str(CACHE), output_dir=str(OUTPUT),
           device='cuda', orca_executable=str(PROJECT / 'tools/orca'))
CONFIG = PROJECT / 'configs/cloud-runtime.yaml'
CONFIG.write_text(yaml.safe_dump(cfg))
print(CONFIG.read_text())


In [ ]:
def run(command, *extra):
    subprocess.run([sys.executable, '-m', 'sebot_mls.cli', command, '--config', str(CONFIG), *extra], check=True)
if not (RAW_DATA / 'features.pt').exists():
    run('download')


**Preprocess once.** Exact ego statistics, centralities and entropy trees can take substantial CPU time. A GPU does not accelerate all of this stage. Stop here after saving the cache if the session deadline is close, and train from the saved cache in a new session.

In [ ]:
run('prepare')


Train with the paper settings; checkpoint selection uses validation loss. Evaluation is a separate command and does not influence training. On Kaggle use Save & Run All. On Colab outputs below persist in Drive.

In [ ]:
run('train', *(['--resume'] if (OUTPUT / 'latest.pt').exists() else []))


In [ ]:
run('evaluate')
print((OUTPUT / 'test_metrics.json').read_text())


In [ ]:
# Bundle source settings and outputs for the local demonstration.
# Copy raw data and the prepared cache separately if they are not already local.
import shutil
result_zip = shutil.make_archive(str(PERSIST / 'sebot-training-outputs'), 'zip', OUTPUT)
print(result_zip)


In [ ]:
from google.colab import files
files.download(result_zip)
